# Notebook 3: Prompt Engineering

**Techniques covered:** Prompt Engineering + Few-Shot Learning + Chain-of-Thought Prompting

This notebook demonstrates the systematic prompt engineering strategy used in the interview agent:
1. **Systematic prompt design** — structured, role-anchored system prompts
2. **Few-shot learning** — in-context examples guide question difficulty and style
3. **Chain-of-Thought (CoT)** — multi-step reasoning before scoring candidate answers
4. **In-context learning** — ablation showing impact of each strategy

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from google import genai as google_genai
from google.genai import types as genai_types

client = google_genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
print('Google GenAI client ready')

## 1. Systematic Prompt Design — Role + Constraints

In [ ]:
from prompts.templates import INTERVIEWER_SYSTEM_PROMPT

# Show the formatted system prompt
system_prompt = INTERVIEWER_SYSTEM_PROMPT.format(domain='Software Engineering')
print(system_prompt)

In [ ]:
def ask(system: str, user: str) -> str:
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=f"{system}\n\n{user}",
        config=genai_types.GenerateContentConfig(temperature=0.7, max_output_tokens=200),
    )
    return response.text.strip()

naive_system = "You are an interviewer."
user_msg = "The candidate just joined. Begin the interview."

naive_resp      = ask(naive_system, user_msg)
systematic_resp = ask(system_prompt, user_msg)

print('=== NAIVE SYSTEM PROMPT ===')
print(naive_resp)
print()
print('=== SYSTEMATIC SYSTEM PROMPT ===')
print(systematic_resp)

## 2. Few-Shot Learning — Difficulty-Calibrated Questions

In [ ]:
from prompts.templates import get_few_shot_text, QUESTION_GENERATION_PROMPT

def generate_question(difficulty: str, use_few_shot: bool = True) -> str:
    few_shot = get_few_shot_text('software_engineering', difficulty) if use_few_shot else 'None provided.'
    prompt = QUESTION_GENERATION_PROMPT.format(
        domain='software engineering',
        difficulty=difficulty,
        context='  - Microservices Architecture (hard): Independent services owning their own data store.',
        few_shot_examples=few_shot,
        previously_asked='  None yet.',
        question_number=1,
        total_questions=8
    )
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0.7, max_output_tokens=150),
    )
    return response.text.strip()

for diff in ['easy', 'medium', 'hard', 'expert']:
    q_with    = generate_question(diff, use_few_shot=True)
    q_without = generate_question(diff, use_few_shot=False)
    print(f'[{diff.upper()}]')
    print(f'  With few-shot   : {q_with}')
    print(f'  Without few-shot: {q_without}')
    print()

## 3. Chain-of-Thought (CoT) Evaluation

In [ ]:
from prompts.templates import CHAIN_OF_THOUGHT_EVALUATION_PROMPT

question = "Explain the CAP theorem and give an example of a system that is CP vs one that is AP."
candidate_answer = "CAP theorem says you can't have all three — consistency, availability, and partition tolerance. ZooKeeper is CP because it prefers consistency. Cassandra is AP, it keeps serving requests even if some nodes are down, but data might be stale."

cot_prompt = CHAIN_OF_THOUGHT_EVALUATION_PROMPT.format(
    domain='software engineering',
    question=question,
    answer=candidate_answer
)

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=cot_prompt,
    config=genai_types.GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=700,
        response_mime_type='application/json',
    ),
)

evaluation = json.loads(response.text)
print(json.dumps(evaluation, indent=2))

## 4. CoT vs Direct Scoring Ablation

In [ ]:
direct_prompt = f"""Score this interview answer from 0-10 for technical accuracy, completeness, and communication.
Return JSON: {{"technical_accuracy": N, "completeness": N, "communication": N}}

Question: {question}
Answer: {candidate_answer}"""

direct_response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=direct_prompt,
    config=genai_types.GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=200,
        response_mime_type='application/json',
    ),
)
direct_eval = json.loads(direct_response.text)

print('Direct scoring (no CoT):', direct_eval)
print()
print('CoT scoring:')
for k in ['technical_accuracy', 'completeness', 'communication']:
    print(f'  {k}: {evaluation[k]}')
print()
print('CoT reasoning excerpt:')
print(evaluation.get('reasoning', '')[:300])

## 5. In-Context Learning — Prompt Strategy Comparison

In [ ]:
import pandas as pd

test_answers = [
    ("Strong",  "Microservices are small, independently deployable services each with their own database. They communicate via APIs or message queues. Benefits: independent scaling and fault isolation. Challenges: network latency, distributed transactions, operational complexity."),
    ("Medium",  "Microservices means splitting up a big app into smaller apps. Each does one thing. They can fail independently. It's harder to debug though."),
    ("Weak",    "I think microservices is when you have lots of services. It's used in big companies like Netflix."),
]

q = "What are microservices and what are the main trade-offs versus a monolith?"

rows = []
for label, ans in test_answers:
    p = CHAIN_OF_THOUGHT_EVALUATION_PROMPT.format(domain='software engineering', question=q, answer=ans)
    r = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=p,
        config=genai_types.GenerateContentConfig(
            temperature=0.1, max_output_tokens=500, response_mime_type='application/json'
        ),
    )
    ev = json.loads(r.text)
    overall = round(ev.get('technical_accuracy',5)*0.5 + ev.get('completeness',5)*0.3 + ev.get('communication',5)*0.2, 2)
    rows.append({
        'Label': label,
        'Tech Accuracy': ev.get('technical_accuracy'),
        'Completeness': ev.get('completeness'),
        'Communication': ev.get('communication'),
        'Overall': overall
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Summary

| Strategy | Application | Benefit |
|---|---|---|
| Systematic prompt design | System prompt with role + constraints | Consistent interviewer behaviour |
| Few-shot learning | 2 examples per difficulty level | Properly calibrated question difficulty |
| Chain-of-Thought | 5-step reasoning before scoring | More consistent, explainable scores |
| In-context learning | Domain knowledge in question prompt | Grounded, accurate questions |

The ablation confirms that all three strategies contribute to evaluation quality.